In [1]:
import pandas as pd

df = pd.read_csv("../data/raw/content.csv")

df.head()

,content_id,title,type,overview,genres,cast,language,rating,popularity,release_date
0,550,Fight Club,movie,An insomniac office worker and a soap maker fo...,Drama|Thriller,Brad Pitt|Edward Norton|Helena Bonham Carter,en,8.4,61.4,1999-10-15
1,1399,Game of Thrones,tv,Nine noble families fight for control over the...,Drama|Fantasy|Action,Emilia Clarke|Kit Harington|Peter Dinklage,en,8.5,369.5,2011-04-17


In [3]:
df.shape

(2, 10)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2 entries, 0 to 1
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   content_id    2 non-null      int64  
 1   title         2 non-null      str    
 2   type          2 non-null      str    
 3   overview      2 non-null      str    
 4   genres        2 non-null      str    
 5   cast          2 non-null      str    
 6   language      2 non-null      str    
 7   rating        2 non-null      float64
 8   popularity    2 non-null      float64
 9   release_date  2 non-null      str    
dtypes: float64(2), int64(1), str(7)
memory usage: 292.0 bytes


In [5]:
df.isnull().sum()

content_id      0
title           0
type            0
overview        0
genres          0
cast            0
language        0
rating          0
popularity      0
release_date    0
dtype: int64

In [6]:
df.describe()

,content_id,rating,popularity
count,2.000000,2.000000,2.000000
mean,974.500000,8.450000,215.450000
std,600.333657,0.070711,217.859599
min,550.000000,8.400000,61.400000
25%,762.250000,8.425000,138.425000
50%,974.500000,8.450000,215.450000
75%,1186.750000,8.475000,292.475000
max,1399.000000,8.500000,369.500000


In [10]:
df.duplicated().sum()

np.int64(0)

In [11]:
df["content_id"].duplicated().sum()

np.int64(0)

In [12]:
df["type"].value_counts()

type
movie    1
tv       1
Name: count, dtype: int64

In [13]:
text_columns = ["title", "overview", "genres", "cast"]

for column in text_columns:
    print(f"\n{column}:")
    print(df[column].tolist())


title:
['Fight Club', 'Game of Thrones']

overview:
['An insomniac office worker and a soap maker form an underground fight club.', 'Nine noble families fight for control over the lands of Westeros.']

genres:
['Drama|Thriller', 'Drama|Fantasy|Action']

cast:
['Brad Pitt|Edward Norton|Helena Bonham Carter', 'Emilia Clarke|Kit Harington|Peter Dinklage']


In [14]:
print("Rating range:", df["rating"].min(), "to", df["rating"].max())
print("Popularity range:", df["popularity"].min(), "to", df["popularity"].max())

Rating range: 8.4 to 8.5
Popularity range: 61.4 to 369.5


In [15]:
text_columns = ["title", "overview", "genres", "cast"]

df["content_text"] = df[text_columns].fillna("").agg(" ".join, axis=1)

In [16]:
df[["title", "content_text"]]

,title,content_text
0,Fight Club,Fight Club An insomniac office worker and a so...
1,Game of Thrones,Game of Thrones Nine noble families fight for ...


In [17]:
df["content_text"].iloc[0]

'Fight Club An insomniac office worker and a soap maker form an underground fight club. Drama|Thriller Brad Pitt|Edward Norton|Helena Bonham Carter'

In [18]:
df["content_text"].iloc[1]

'Game of Thrones Nine noble families fight for control over the lands of Westeros. Drama|Fantasy|Action Emilia Clarke|Kit Harington|Peter Dinklage'

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df["content_text"])

In [20]:
tfidf_matrix.shape

(2, 40)

In [21]:
vectorizer.get_feature_names_out()

array(['action', 'an', 'and', 'bonham', 'brad', 'carter', 'clarke',
       'club', 'control', 'dinklage', 'drama', 'edward', 'emilia',
       'families', 'fantasy', 'fight', 'for', 'form', 'game', 'harington',
       'helena', 'insomniac', 'kit', 'lands', 'maker', 'nine', 'noble',
       'norton', 'of', 'office', 'over', 'peter', 'pitt', 'soap', 'the',
       'thriller', 'thrones', 'underground', 'westeros', 'worker'],
      dtype=object)

In [22]:
tfidf_matrix.toarray()

array([[0.        , 0.38828569, 0.19414285, 0.19414285, 0.19414285,
        0.19414285, 0.        , 0.38828569, 0.        , 0.        ,
        0.13813423, 0.19414285, 0.        , 0.        , 0.        ,
        0.27626847, 0.        , 0.19414285, 0.        , 0.        ,
        0.19414285, 0.19414285, 0.        , 0.        , 0.19414285,
        0.        , 0.        , 0.19414285, 0.        , 0.19414285,
        0.        , 0.        , 0.19414285, 0.19414285, 0.        ,
        0.19414285, 0.        , 0.19414285, 0.        , 0.19414285],
       [0.20407106, 0.        , 0.        , 0.        , 0.        ,
        0.        , 0.20407106, 0.        , 0.20407106, 0.20407106,
        0.14519824, 0.        , 0.20407106, 0.20407106, 0.20407106,
        0.14519824, 0.20407106, 0.        , 0.20407106, 0.20407106,
        0.        , 0.        , 0.20407106, 0.20407106, 0.        ,
        0.20407106, 0.20407106, 0.        , 0.40814212, 0.        ,
        0.20407106, 0.20407106, 0.        , 0. 

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
similarity_matrix = cosine_similarity(tfidf_matrix)

In [25]:
similarity_matrix

array([[1.        , 0.06017054],
       [0.06017054, 1.        ]])

In [26]:
similarity_scores = list(enumerate(similarity_matrix[0]))

similarity_scores

[(0, np.float64(1.0)), (1, np.float64(0.06017054275477983))]

In [27]:
similarity_scores = sorted(
    similarity_scores,
    key=lambda x: x[1],
    reverse=True
)

similarity_scores

[(0, np.float64(1.0)), (1, np.float64(0.06017054275477983))]

In [28]:
similarity_scores = similarity_scores[1:]

In [29]:
similarity_scores

[(1, np.float64(0.06017054275477983))]

In [30]:
for index, score in similarity_scores:
    print(df.iloc[index]["title"], score)

Game of Thrones 0.06017054275477983


In [31]:
def recommend(title, top_k=5):
    # Find the content matching the given title
    matches = df[df["title"].str.lower() == title.lower()]

    if matches.empty:
        return f"'{title}' not found in dataset."

    # Get its dataframe index
    content_index = matches.index[0]

    # Get similarity scores against every item
    similarity_scores = list(
        enumerate(similarity_matrix[content_index])
    )

    # Sort from most similar to least similar
    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    # Remove the content itself and keep Top-K
    similarity_scores = [
        (index, score)
        for index, score in similarity_scores
        if index != content_index
    ][:top_k]

    # Build readable results
    recommendations = []

    for index, score in similarity_scores:
        recommendations.append({
            "content_id": int(df.iloc[index]["content_id"]),
            "title": df.iloc[index]["title"],
            "type": df.iloc[index]["type"],
            "similarity_score": round(float(score), 4)
        })

    return recommendations

In [32]:
recommend("Fight Club")

[{'content_id': 1399,
  'title': 'Game of Thrones',
  'type': 'tv',
  'similarity_score': 0.0602}]

In [33]:
recommend("Game of Thrones")

[{'content_id': 550,
  'title': 'Fight Club',
  'type': 'movie',
  'similarity_score': 0.0602}]

In [34]:
recommend("Interstellar")

"'Interstellar' not found in dataset."

In [35]:
import pandas as pd

df = pd.read_csv("../data/raw/content.csv")

print(df.shape)
df.head()

(20, 10)


,content_id,title,type,overview,genres,cast,language,rating,popularity,release_date
0,969681,Spider-Man: Brand New Day,movie,Fighting crime full-time as Spider-Man in a wo...,Science Fiction|Action|Adventure,Tom Holland|Zendaya|Mark Ruffalo|Jon Bernthal|...,en,7.867,564.5909,2026-07-29
1,1599191,Renegade Immortal: Battle of the Immortal Slayer,movie,Trapped in a fatal pursuit within the Celestia...,Animation|Action|Adventure|Fantasy,Shi Zekun|Li Jiaxiang|Liu Sicen|Shuai Guan|Si ...,zh,7.800,545.7147,2026-10-01
2,1423191,Resident Evil,movie,Medical courier Bryan unwittingly finds himsel...,Horror|Science Fiction|Adventure,Austin Abrams|Paul Walter Hauser|Kali Reis|Zac...,en,7.313,433.5555,2026-09-16
3,1204680,Coyote vs. Acme,movie,After Acme products fail him one too many time...,Comedy|Adventure|Family,Will Forte|Lana Condor|John Cena|Tone Bell|Mar...,en,7.530,290.8185,2026-08-17
4,1368337,The Odyssey,movie,"Odysseus, the legendary King of Ithaca, embark...",Adventure|Action|Fantasy,Matt Damon|Tom Holland|Anne Hathaway|Robert Pa...,en,8.000,298.6091,2026-07-15


In [36]:
text_columns = ["title", "overview", "genres", "cast"]

df["content_text"] = (
    df[text_columns]
    .fillna("")
    .agg(" ".join, axis=1)
)

In [37]:
df[["title", "content_text"]].head()

,title,content_text
0,Spider-Man: Brand New Day,Spider-Man: Brand New Day Fighting crime full-...
1,Renegade Immortal: Battle of the Immortal Slayer,Renegade Immortal: Battle of the Immortal Slay...
2,Resident Evil,Resident Evil Medical courier Bryan unwittingl...
3,Coyote vs. Acme,Coyote vs. Acme After Acme products fail him o...
4,The Odyssey,"The Odyssey Odysseus, the legendary King of It..."


In [38]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(
    df["content_text"]
)

print(tfidf_matrix.shape)

(20, 874)


In [39]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print(similarity_matrix.shape)

(20, 20)


In [40]:
def recommend(title, top_k=5):
    matches = df[df["title"].str.lower() == title.lower()]

    if matches.empty:
        return f"'{title}' not found in dataset."

    content_index = matches.index[0]

    similarity_scores = list(
        enumerate(similarity_matrix[content_index])
    )

    similarity_scores = sorted(
        similarity_scores,
        key=lambda x: x[1],
        reverse=True
    )

    similarity_scores = [
        (index, score)
        for index, score in similarity_scores
        if index != content_index
    ][:top_k]

    recommendations = []

    for index, score in similarity_scores:
        recommendations.append({
            "content_id": int(df.iloc[index]["content_id"]),
            "title": df.iloc[index]["title"],
            "type": df.iloc[index]["type"],
            "similarity_score": round(float(score), 4)
        })

    return recommendations

In [41]:
df["title"].tolist()

['Spider-Man: Brand New Day',
 'Renegade Immortal: Battle of the Immortal Slayer',
 'Resident Evil',
 'Coyote vs. Acme',
 'The Odyssey',
 'The Fix',
 'Runner',
 'The End of Oak Street',
 'After Impact',
 'The Love Hypothesis',
 'Digger',
 'The Mongoose',
 'Colony',
 'Mutiny',
 'Resident Evil: Welcome to Raccoon City',
 'Toy Story 5',
 'UNABOMBER',
 'Heart of the Beast',
 'The Uprising',
 'Obsession']

In [43]:
recommend("The Odyssey", top_k=5)

[{'content_id': 1248832,
  'title': 'Digger',
  'type': 'movie',
  'similarity_score': 0.151},
 {'content_id': 1751701,
  'title': 'After Impact',
  'type': 'movie',
  'similarity_score': 0.1278},
 {'content_id': 969681,
  'title': 'Spider-Man: Brand New Day',
  'type': 'movie',
  'similarity_score': 0.1268},
 {'content_id': 460458,
  'title': 'Resident Evil: Welcome to Raccoon City',
  'type': 'movie',
  'similarity_score': 0.1247},
 {'content_id': 1599191,
  'title': 'Renegade Immortal: Battle of the Immortal Slayer',
  'type': 'movie',
  'similarity_score': 0.1012}]

In [44]:
tfidf_matrix.shape

(20, 874)

In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/content.csv")

print(df.shape)
print(df["type"].value_counts())

(60, 10)
type
movie    20
tv       20
anime    20
Name: count, dtype: int64


In [3]:
import pandas as pd

df = pd.read_csv("../data/raw/content.csv")

print("Shape:", df.shape)

print("\nContent distribution:")
print(df["type"].value_counts())

print("\nDuplicate content IDs:")
print(df["content_id"].duplicated().sum())

print("\nDuplicate title + type:")
print(df.duplicated(subset=["title", "type"]).sum())

print("\nMissing values:")
print(df.isnull().sum())

Shape: (120, 10)

Content distribution:
type
movie    40
tv       40
anime    40
Name: count, dtype: int64

Duplicate content IDs:
2

Duplicate title + type:
2

Missing values:
content_id      0
title           0
type            0
overview        0
genres          1
cast            8
language        0
rating          0
popularity      0
release_date    0
dtype: int64


In [4]:
required_columns = [
    "content_id",
    "title",
    "type",
    "overview",
    "genres",
    "cast"
]

print("\nMissing values in recommendation fields:")
print(df[required_columns].isna().sum())


Missing values in recommendation fields:
content_id    0
title         0
type          0
overview      0
genres        1
cast          8
dtype: int64


In [5]:
duplicates = df[df.duplicated(
    subset=["content_id", "type"],
    keep=False
)].sort_values("content_id")

duplicates[["content_id", "title", "type"]]

,content_id,title,type


In [6]:
duplicates = df[
    df["content_id"].duplicated(keep=False)
].sort_values("content_id")

duplicates[["content_id", "title", "type"]]

,content_id,title,type
72,65733,Doraemon,tv
81,65733,Doraemon,anime
47,70998,Souryo to Majiwaru Shikiyoku no Yoru ni...,tv
80,70998,Souryo to Majiwaru Shikiyoku no Yoru ni...,anime


In [7]:
tv_ids = set(df[df["type"] == "tv"]["content_id"])
anime_ids = set(df[df["type"] == "anime"]["content_id"])

overlap = tv_ids & anime_ids

print("TV/Anime overlap:", len(overlap))
print("Overlapping IDs:", overlap)

TV/Anime overlap: 2
Overlapping IDs: {65733, 70998}


In [8]:
missing_text = df[
    df["genres"].isna() | df["cast"].isna()
]

missing_text[
    ["content_id", "title", "type", "genres", "cast"]
]

,content_id,title,type,genres,cast
50,4304,Xplay,tv,NaN,NaN
57,2224,The Daily Show,tv,News|Comedy,NaN
59,94722,Tagesschau,tv,News,NaN
73,11890,"Goede Tijden, Slechte Tijden",tv,Soap,NaN
75,4656,Raw,tv,Reality,NaN
91,34860,Ninja Boy Rantaro,anime,Comedy|Action & Adventure|Kids|Animation|Family,NaN
96,95479,JUJUTSU KAISEN,anime,Animation|Sci-Fi & Fantasy|Action & Adventure,NaN
115,46392,Duel Masters,anime,Action & Adventure|Animation|Sci-Fi & Fantasy|...,NaN


In [9]:
df = pd.read_csv("../data/raw/content.csv")

print(df[["genres", "cast"]].isna().sum())

genres    1
cast      8
dtype: int64


In [10]:
text_columns = ["title", "overview", "genres", "cast"]

df[text_columns] = df[text_columns].fillna("")

df["content_text"] = df[text_columns].agg(" ".join, axis=1)

df[["title", "content_text"]].head()

,title,content_text
0,JUJUTSU KAISEN,JUJUTSU KAISEN Yuji Itadori is a boy with trem...
1,Secret Mission - Undercover Agents Never Back ...,Secret Mission - Undercover Agents Never Back ...
2,Dragon Ball Z,Dragon Ball Z Now happily married and with a s...
3,Crazy Over His Fingers: Just the Two of Us in ...,Crazy Over His Fingers: Just the Two of Us in ...
4,Ranma1/2,"Ranma1/2 Akane Tendo meets her new fiancé, Ran..."


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

tfidf_matrix = vectorizer.fit_transform(df["content_text"])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (118, 3618)


In [12]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(tfidf_matrix)

print("Similarity matrix shape:", similarity_matrix.shape)

Similarity matrix shape: (118, 118)


In [13]:
import numpy as np

index = df.index[df["title"] == "Dragon Ball Z"][0]

scores = similarity_matrix[index]

print("Content:", df.loc[index, "title"])
print("Self similarity:", scores[index])

Content: Dragon Ball Z
Self similarity: 0.9999999999999986


In [14]:
similar_indices = np.argsort(scores)[::-1]

for i in similar_indices[1:6]:
    print(
        df.loc[i, "title"],
        "→",
        round(scores[i], 3)
    )

Ranma1/2 → 0.074
Ranma ½ → 0.061
Doraemon → 0.059
Fullmetal Alchemist: Brotherhood → 0.058
JoJo's Bizarre Adventure → 0.055


In [15]:
feature_names = vectorizer.get_feature_names_out()

print("Number of features:", len(feature_names))
print("\nFirst 50 features:")
print(feature_names[:50])

Number of features: 3618

First 50 features:
['000' '10' '100' '12' '14th' '1962' '1963' '1972' '1973' '1982' '1992'
 '1995' '1998' '2002' '2006' '2013' '2021' '22nd' '23' '34' '42' '50' '67'
 '90' 'aadukalam' 'aaron' 'abandoned' 'abbott' 'abilities' 'ability'
 'able' 'about' 'abrams' 'absolutely' 'academy' 'acar' 'accident'
 'accidentally' 'accompanied' 'accused' 'achar' 'achieve' 'achieved'
 'ackles' 'acme' 'acquired' 'action' 'actions' 'actually' 'adam']


In [16]:
def recommend(title, top_k=5):
    # Find the index of the requested content
    matches = df.index[df["title"].str.lower() == title.lower()]

    if len(matches) == 0:
        return f"Content '{title}' not found."

    index = matches[0]

    # Get similarity scores for this content
    scores = similarity_matrix[index]

    # Sort from highest to lowest
    similar_indices = np.argsort(scores)[::-1]

    # Remove the content itself and select top K
    recommendations = []

    for i in similar_indices:
        if i == index:
            continue

        recommendations.append({
            "title": df.loc[i, "title"],
            "type": df.loc[i, "type"],
            "similarity_score": round(scores[i], 3)
        })

        if len(recommendations) == top_k:
            break

    return pd.DataFrame(recommendations)

In [17]:
recommend("Dragon Ball Z", 5)

,title,type,similarity_score
0,Ranma1/2,anime,0.074
1,Ranma ½,anime,0.061
2,Doraemon,anime,0.059
3,Fullmetal Alchemist: Brotherhood,anime,0.058
4,JoJo's Bizarre Adventure,anime,0.055


In [18]:
recommend("Fight Club", 5)

"Content 'Fight Club' not found."

In [19]:
recommend("Game of Thrones", 5)

"Content 'Game of Thrones' not found."

In [20]:
print(df[df["title"].str.contains(
    "Fight Club|Game of Thrones",
    case=False,
    na=False
)][["content_id", "title", "type"]])

Empty DataFrame
Columns: [content_id, title, type]
Index: []


In [21]:
df[["content_id", "title", "type"]].sample(10)

,content_id,title,type
47,61818,Late Night with Seth Meyers,tv
111,1339713,Obsession,movie
88,1482547,Zip Wire,movie
116,1283515,Verity,movie
107,1375646,Colony,movie
41,275102,The Scandal,tv
29,220150,Pokémon Horizons,anime
102,1751701,After Impact,movie
76,79744,The Rookie,tv
57,82873,The Kelly Clarkson Show,tv


In [22]:
recommend("Obsession", 5)

,title,type,similarity_score
0,Watch What Happens Live with Andy Cohen,tv,0.082
1,After Impact,movie,0.072
2,Pinocchio: Unstrung,movie,0.068
3,Bleach,anime,0.061
4,Monster,anime,0.061


In [23]:
index = df.index[df["title"] == "Dragon Ball Z"][0]

vector = tfidf_matrix[index].toarray().flatten()

top_indices = vector.argsort()[::-1][:15]

for i in top_indices:
    print(feature_names[i], "→", round(vector[i], 3))

yanami → 0.167
suzuoki → 0.167
ryo → 0.167
masako → 0.167
tatsuta → 0.167
nozawa → 0.167
invaders → 0.167
toshio → 0.167
extraterrestrial → 0.167
furukawa → 0.167
goku → 0.167
married → 0.167
hirotaka → 0.167
horikawa → 0.167
naoki → 0.167


In [24]:
print("Dragon Ball Z genres:")
print(df.loc[index, "genres"])

print("\nDragon Ball Z cast:")
print(df.loc[index, "cast"])

Dragon Ball Z genres:
Animation|Sci-Fi & Fantasy|Action & Adventure

Dragon Ball Z cast:
Masako Nozawa|Ryo Horikawa|Toshio Furukawa|Mayumi Tanaka|Naoki Tatsuta|Ryusei Nakao|Joji Yanami|Naoko Watanabe|Hirotaka Suzuoki|Miki Ito


In [26]:
df["content_text_weighted"] = (
    df["title"] + " " +
    df["overview"] + " " +
    df["overview"] + " " +
    df["genres"] + " " +
    df["genres"] + " " +
    df["genres"] + " " +
    df["cast"]
)

In [27]:
df[["title", "content_text_weighted"]].head(3)

,title,content_text_weighted
0,JUJUTSU KAISEN,JUJUTSU KAISEN Yuji Itadori is a boy with trem...
1,Secret Mission - Undercover Agents Never Back ...,Secret Mission - Undercover Agents Never Back ...
2,Dragon Ball Z,Dragon Ball Z Now happily married and with a s...


In [28]:
weighted_vectorizer = TfidfVectorizer()

weighted_tfidf_matrix = weighted_vectorizer.fit_transform(
    df["content_text_weighted"]
)

print("Weighted TF-IDF shape:", weighted_tfidf_matrix.shape)

Weighted TF-IDF shape: (118, 3618)


In [29]:
weighted_similarity_matrix = cosine_similarity(weighted_tfidf_matrix)

print("Weighted similarity matrix shape:",
      weighted_similarity_matrix.shape)

Weighted similarity matrix shape: (118, 118)


In [30]:
index = df.index[df["title"] == "Dragon Ball Z"][0]

In [31]:
normal_scores = similarity_matrix[index]
weighted_scores = weighted_similarity_matrix[index]

normal_indices = np.argsort(normal_scores)[::-1]
weighted_indices = np.argsort(weighted_scores)[::-1]

print("=== Normal TF-IDF ===")

count = 0
for i in normal_indices:
    if i == index:
        continue

    print(
        df.loc[i, "title"],
        "→",
        round(normal_scores[i], 3)
    )

    count += 1
    if count == 5:
        break


print("\n=== Weighted TF-IDF ===")

count = 0
for i in weighted_indices:
    if i == index:
        continue

    print(
        df.loc[i, "title"],
        "→",
        round(weighted_scores[i], 3)
    )

    count += 1
    if count == 5:
        break

=== Normal TF-IDF ===
Ranma1/2 → 0.074
Ranma ½ → 0.061
Doraemon → 0.059
Fullmetal Alchemist: Brotherhood → 0.058
JoJo's Bizarre Adventure → 0.055

=== Weighted TF-IDF ===
JoJo's Bizarre Adventure → 0.16
Doraemon → 0.145
Pokémon → 0.143
Hunter x Hunter → 0.142
Pokémon Horizons → 0.142


In [32]:
test_titles = [
    "Dragon Ball Z",
    "Doraemon",
    "Bleach",
    "Fullmetal Alchemist: Brotherhood",
    "JoJo's Bizarre Adventure"
]

for title in test_titles:
    index = df.index[df["title"] == title][0]
    scores = weighted_similarity_matrix[index]

    similar_indices = np.argsort(scores)[::-1]

    print(f"\n=== {title} ===")

    count = 0
    for i in similar_indices:
        if i == index:
            continue

        print(
            df.loc[i, "title"],
            "→",
            round(scores[i], 3)
        )

        count += 1
        if count == 5:
            break


=== Dragon Ball Z ===
JoJo's Bizarre Adventure → 0.16
Doraemon → 0.145
Pokémon → 0.143
Hunter x Hunter → 0.142
Pokémon Horizons → 0.142

=== Doraemon ===
Doraemon → 0.232
JoJo's Bizarre Adventure → 0.181
Pokémon Horizons → 0.161
Family Guy → 0.151
Dragon Ball Z → 0.145

=== Bleach ===
Re:ZERO -Starting Life in Another World- → 0.182
JoJo's Bizarre Adventure → 0.179
Pokémon → 0.175
JUJUTSU KAISEN → 0.172
Mushoku Tensei: Jobless Reincarnation → 0.166

=== Fullmetal Alchemist: Brotherhood ===
Naruto Shippūden → 0.178
Supernatural → 0.169
JoJo's Bizarre Adventure → 0.156
Re:ZERO -Starting Life in Another World- → 0.15
One Piece → 0.14

=== JoJo's Bizarre Adventure ===
Pokémon Horizons → 0.205
Doraemon → 0.181
Bleach → 0.179
Naruto Shippūden → 0.174
Naruto → 0.169


In [33]:
## 7. Model Evaluation

We compare the standard TF-IDF model with the weighted TF-IDF model
to determine which representation produces more relevant recommendations.

SyntaxError: invalid syntax (3830165848.py, line 3)

In [34]:
df[df["title"].str.contains(
    "Dragon Ball",
    case=False,
    na=False
)][[
    "content_id",
    "title",
    "type",
    "genres"
]]

,content_id,title,type,genres
2,12971,Dragon Ball Z,anime,Animation|Sci-Fi & Fantasy|Action & Adventure


In [35]:
df[["content_id", "title", "type"]].sample(
    10,
    random_state=42
)

,content_id,title,type
56,1622,Supernatural,tv
89,1607127,One Last Shot,movie
4,259140,Ranma1/2,anime
45,549,Law & Order,tv
26,72517,Classroom of the Elite,anime
116,1283515,Verity,movie
69,4614,NCIS,tv
10,37854,One Piece,anime
90,1108427,Moana,movie
73,4656,Raw,tv


In [36]:
evaluation_titles = df.loc[
    [56, 89, 4, 45, 26, 116, 69, 10, 90, 73],
    "title"
].tolist()

for title in evaluation_titles:
    index = df.index[df["title"] == title][0]
    scores = weighted_similarity_matrix[index]

    similar_indices = np.argsort(scores)[::-1]

    print(f"\n=== {title} ===")

    count = 0
    for i in similar_indices:
        if i == index:
            continue

        print(
            f"{df.loc[i, 'title']} "
            f"({df.loc[i, 'type']}) → "
            f"{scores[i]:.3f}"
        )

        count += 1
        if count == 5:
            break


=== Supernatural ===
Pokémon Horizons (anime) → 0.184
Monogatari (anime) → 0.170
Fullmetal Alchemist: Brotherhood (anime) → 0.169
The End of Oak Street (movie) → 0.164
Lanterns (tv) → 0.147

=== One Last Shot ===
The Fix (movie) → 0.116
The Mongoose (movie) → 0.099
UNABOMBER (movie) → 0.092
NCIS (tv) → 0.085
War of the Worlds (movie) → 0.070

=== Ranma1/2 ===
Ranma ½ (anime) → 0.346
Dragon Ball Z (anime) → 0.134
Pokémon (anime) → 0.124
Re:ZERO -Starting Life in Another World- (anime) → 0.105
Mushoku Tensei: Jobless Reincarnation (anime) → 0.094

=== Law & Order ===
Law & Order: Special Victims Unit (tv) → 0.189
CSI: Crime Scene Investigation (tv) → 0.157
Criminal Minds (tv) → 0.123
Detective Conan (anime) → 0.114
The Mentalist (tv) → 0.112

=== Classroom of the Elite ===
Law & Order: Special Victims Unit (tv) → 0.103
Criminal Minds (tv) → 0.084
Law & Order (tv) → 0.083
Adam's Sweet Agony (anime) → 0.076
Ninja Boy Rantaro (anime) → 0.076

=== Verity ===
The Odyssey (movie) → 0.088
War 

In [37]:
def genre_overlap(seed_index, recommended_index):
    seed_genres = set(
        str(df.loc[seed_index, "genres"]).split("|")
    )

    recommended_genres = set(
        str(df.loc[recommended_index, "genres"]).split("|")
    )

    if not seed_genres or not recommended_genres:
        return 0

    return len(seed_genres & recommended_genres) / len(seed_genres)

In [38]:
seed_index = df.index[df["title"] == "Dragon Ball Z"][0]
recommended_index = df.index[
    df["title"] == "JoJo's Bizarre Adventure"
][0]

score = genre_overlap(seed_index, recommended_index)

print("Genre overlap:", round(score, 3))

Genre overlap: 1.0


In [39]:
evaluation_indices = [56, 89, 4, 45, 26, 116, 69, 10, 90, 73]

weighted_overlap_scores = []

for index in evaluation_indices:
    scores = weighted_similarity_matrix[index]
    similar_indices = np.argsort(scores)[::-1]

    count = 0

    for i in similar_indices:
        if i == index:
            continue

        overlap = genre_overlap(index, i)
        weighted_overlap_scores.append(overlap)

        count += 1

        if count == 5:
            break

average_weighted_overlap = np.mean(weighted_overlap_scores)

print(
    "Average genre overlap (Weighted TF-IDF):",
    round(average_weighted_overlap, 3)
)

Average genre overlap (Weighted TF-IDF): 0.51


In [40]:
normal_overlap_scores = []

for index in evaluation_indices:
    scores = similarity_matrix[index]
    similar_indices = np.argsort(scores)[::-1]

    count = 0

    for i in similar_indices:
        if i == index:
            continue

        overlap = genre_overlap(index, i)
        normal_overlap_scores.append(overlap)

        count += 1

        if count == 5:
            break

average_normal_overlap = np.mean(normal_overlap_scores)

print(
    "Average genre overlap (Standard TF-IDF):",
    round(average_normal_overlap, 3)
)

Average genre overlap (Standard TF-IDF): 0.412


In [41]:
## 8. Personalized Recommendation Experiment

In [42]:
user_history = [
    "Dragon Ball Z",
    "Bleach",
    "One Piece"
]

print("User history:")
for title in user_history:
    print("-", title)

User history:
- Dragon Ball Z
- Bleach
- One Piece


In [43]:
user_indices = []

for title in user_history:
    matches = df.index[df["title"].str.lower() == title.lower()]

    if len(matches) == 0:
        print(f"'{title}' not found")
    else:
        user_indices.append(matches[0])

print("User content indices:", user_indices)

User content indices: [2, 21, 10]


In [44]:
user_indices = []

for title in user_history:
    matches = df.index[df["title"].str.lower() == title.lower()]

    if len(matches) == 0:
        print(f"'{title}' not found")
    else:
        user_indices.append(matches[0])

print("User content indices:", user_indices)

User content indices: [2, 21, 10]


In [47]:
user_profile = np.asarray(
    weighted_tfidf_matrix[user_indices].mean(axis=0)
)

print("User profile shape:", user_profile.shape)

User profile shape: (1, 3618)


In [48]:
user_similarity = cosine_similarity(
    user_profile,
    weighted_tfidf_matrix
).flatten()

print("Similarity scores:", user_similarity.shape)

Similarity scores: (118,)


In [49]:
watched_indices = set(user_indices)

ranked_indices = np.argsort(user_similarity)[::-1]

recommendations = []

for i in ranked_indices:
    if i in watched_indices:
        continue

    recommendations.append({
        "content_id": df.loc[i, "content_id"],
        "title": df.loc[i, "title"],
        "type": df.loc[i, "type"],
        "score": round(user_similarity[i], 3)
    })

    if len(recommendations) == 10:
        break

pd.DataFrame(recommendations)

,content_id,title,type,score
0,45790,JoJo's Bizarre Adventure,anime,0.246
1,95479,JUJUTSU KAISEN,anime,0.220
2,60572,Pokémon,anime,0.217
3,31910,Naruto Shippūden,anime,0.217
4,65942,Re:ZERO -Starting Life in Another World-,anime,0.216
5,94664,Mushoku Tensei: Jobless Reincarnation,anime,0.213
6,31911,Fullmetal Alchemist: Brotherhood,anime,0.205
7,45952,Hunter x Hunter,anime,0.204
8,46392,Duel Masters,anime,0.201
9,209867,Frieren: Beyond Journey's End,anime,0.195


In [50]:
mixed_user_history = [
    "Dragon Ball Z",
    "One Piece",
    "Supernatural",
    "Moana"
]

print("Mixed user history:")
for title in mixed_user_history:
    print("-", title)

Mixed user history:
- Dragon Ball Z
- One Piece
- Supernatural
- Moana


In [51]:
mixed_user_indices = []

for title in mixed_user_history:
    matches = df.index[
        df["title"].str.lower() == title.lower()
    ]

    if len(matches) == 0:
        print(f"'{title}' not found")
    else:
        mixed_user_indices.append(matches[0])

print("Mixed user indices:", mixed_user_indices)

Mixed user indices: [2, 10, 56, 90]


In [52]:
mixed_user_profile = np.asarray(
    weighted_tfidf_matrix[mixed_user_indices].mean(axis=0)
)

print("Mixed user profile shape:", mixed_user_profile.shape)

Mixed user profile shape: (1, 3618)


In [53]:
mixed_user_similarity = cosine_similarity(
    mixed_user_profile,
    weighted_tfidf_matrix
).flatten()

print("Mixed user similarity shape:", mixed_user_similarity.shape)

Mixed user similarity shape: (118,)


In [54]:
watched_indices = set(mixed_user_indices)

ranked_indices = np.argsort(mixed_user_similarity)[::-1]

mixed_recommendations = []

for i in ranked_indices:
    if i in watched_indices:
        continue

    mixed_recommendations.append({
        "content_id": df.loc[i, "content_id"],
        "title": df.loc[i, "title"],
        "type": df.loc[i, "type"],
        "score": round(mixed_user_similarity[i], 3)
    })

    if len(mixed_recommendations) == 10:
        break

pd.DataFrame(mixed_recommendations)

,content_id,title,type,score
0,31911,Fullmetal Alchemist: Brotherhood,anime,0.228
1,209867,Frieren: Beyond Journey's End,anime,0.225
2,45790,JoJo's Bizarre Adventure,anime,0.225
3,57706,Ranma ½,anime,0.217
4,220150,Pokémon Horizons,anime,0.206
5,65942,Re:ZERO -Starting Life in Another World-,anime,0.202
6,31910,Naruto Shippūden,anime,0.194
7,94664,Mushoku Tensei: Jobless Reincarnation,anime,0.193
8,1368337,The Odyssey,movie,0.191
9,57911,Doraemon,anime,0.188


In [55]:
df.loc[
    mixed_user_indices,
    ["title", "type", "genres"]
]

,title,type,genres
2,Dragon Ball Z,anime,Animation|Sci-Fi & Fantasy|Action & Adventure
10,One Piece,anime,Action & Adventure|Comedy|Animation
56,Supernatural,tv,Drama|Mystery|Sci-Fi & Fantasy
90,Moana,movie,Family|Fantasy|Comedy|Adventure


In [56]:
from collections import Counter

user_genres = []

for i in mixed_user_indices:
    genres = str(df.loc[i, "genres"]).split("|")
    user_genres.extend(genres)

genre_counts = Counter(user_genres)

genre_counts

Counter({'Animation': 2,
         'Sci-Fi & Fantasy': 2,
         'Action & Adventure': 2,
         'Comedy': 2,
         'Drama': 1,
         'Mystery': 1,
         'Family': 1,
         'Fantasy': 1,
         'Adventure': 1})

In [57]:
total_genres = sum(genre_counts.values())

genre_preferences = {
    genre: round(count / total_genres, 3)
    for genre, count in genre_counts.items()
}

genre_preferences

{'Animation': 0.154,
 'Sci-Fi & Fantasy': 0.154,
 'Action & Adventure': 0.154,
 'Comedy': 0.154,
 'Drama': 0.077,
 'Mystery': 0.077,
 'Family': 0.077,
 'Fantasy': 0.077,
 'Adventure': 0.077}

In [58]:
def genre_preference_score(item_genres, genre_preferences):
    genres = set(str(item_genres).split("|"))

    score = sum(
        genre_preferences.get(genre, 0)
        for genre in genres
    )

    return score

In [59]:
test_index = 0

score = genre_preference_score(
    df.loc[test_index, "genres"],
    genre_preferences
)

print(df.loc[test_index, "title"])
print("Genre score:", round(score, 3))

JUJUTSU KAISEN
Genre score: 0.462


In [60]:
df["genre_score"] = df["genres"].apply(
    lambda genres: genre_preference_score(
        genres,
        genre_preferences
    )
)

print(df["genre_score"].describe())

count    118.000000
mean       0.229695
std        0.182599
min        0.000000
25%        0.077000
50%        0.154000
75%        0.308000
max        0.693000
Name: genre_score, dtype: float64


In [61]:
df[
    ["title", "type", "genres", "genre_score"]
].sort_values(
    "genre_score",
    ascending=False
).head(10)

,title,type,genres,genre_score
12,Hunter x Hunter,anime,Sci-Fi & Fantasy|Action & Adventure|Animation|...,0.693
9,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.693
17,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.693
5,Ranma ½,anime,Comedy|Action & Adventure|Animation|Sci-Fi & F...,0.616
15,Frieren: Beyond Journey's End,anime,Animation|Action & Adventure|Drama|Sci-Fi & Fa...,0.539
20,Re:ZERO -Starting Life in Another World-,anime,Animation|Mystery|Action & Adventure|Sci-Fi & ...,0.539
35,Ninja Boy Rantaro,anime,Comedy|Action & Adventure|Kids|Animation|Family,0.539
27,Fullmetal Alchemist: Brotherhood,anime,Animation|Action & Adventure|Sci-Fi & Fantasy|...,0.539
85,Minions & Monsters,movie,Adventure|Animation|Comedy|Family|Fantasy,0.539
78,Zootopia 2,movie,Animation|Adventure|Comedy|Mystery|Family,0.539


In [62]:
comparison = df[
    ["title", "type", "genre_score"]
].copy()

comparison["tfidf_score"] = mixed_user_similarity

comparison = comparison[
    ~comparison.index.isin(mixed_user_indices)
]

comparison.sort_values(
    "tfidf_score",
    ascending=False
).head(10)

,title,type,genre_score,tfidf_score
27,Fullmetal Alchemist: Brotherhood,anime,0.539,0.228493
15,Frieren: Beyond Journey's End,anime,0.539,0.224990
19,JoJo's Bizarre Adventure,anime,0.462,0.224818
5,Ranma ½,anime,0.616,0.217340
29,Pokémon Horizons,anime,0.462,0.205554
20,Re:ZERO -Starting Life in Another World-,anime,0.539,0.201516
11,Naruto Shippūden,anime,0.462,0.194340
18,Mushoku Tensei: Jobless Reincarnation,anime,0.462,0.192718
98,The Odyssey,movie,0.154,0.190614
9,Doraemon,anime,0.693,0.187593


In [63]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

df["tfidf_normalized"] = scaler.fit_transform(
    mixed_user_similarity.reshape(-1, 1)
).flatten()

df["genre_normalized"] = scaler.fit_transform(
    df[["genre_score"]]
).flatten()

print(
    df[
        ["title", "tfidf_normalized", "genre_normalized"]
    ].head()
)

                                               title  tfidf_normalized  \
0                                     JUJUTSU KAISEN          0.294130   
1  Secret Mission - Undercover Agents Never Back ...          0.134289   
2                                      Dragon Ball Z          0.929375   
3  Crazy Over His Fingers: Just the Two of Us in ...          0.179750   
4                                           Ranma1/2          0.175355   

   genre_normalized  
0          0.666667  
1          0.222222  
2          0.666667  
3          0.222222  
4          0.666667  


In [64]:
df["hybrid_score"] = (
    0.5 * df["tfidf_normalized"] +
    0.5 * df["genre_normalized"]
)

hybrid_recommendations = (
    df[
        ~df.index.isin(mixed_user_indices)
    ][
        ["title", "type", "tfidf_normalized",
         "genre_normalized", "hybrid_score"]
    ]
    .sort_values("hybrid_score", ascending=False)
    .head(10)
)

hybrid_recommendations

,title,type,tfidf_normalized,genre_normalized,hybrid_score
9,Doraemon,anime,0.297841,1.000000,0.648921
12,Hunter x Hunter,anime,0.280366,1.000000,0.640183
17,Doraemon,anime,0.254949,1.000000,0.627475
5,Ranma ½,anime,0.352173,0.888889,0.620531
27,Fullmetal Alchemist: Brotherhood,anime,0.372542,0.777778,0.575160
15,Frieren: Beyond Journey's End,anime,0.366145,0.777778,0.571961
20,Re:ZERO -Starting Life in Another World-,anime,0.323270,0.777778,0.550524
85,Minions & Monsters,movie,0.255999,0.777778,0.516888
19,JoJo's Bizarre Adventure,anime,0.365831,0.666667,0.516249
29,Pokémon Horizons,anime,0.330646,0.666667,0.498656


In [65]:
weights = [
    (0.8, 0.2),
    (0.7, 0.3),
    (0.6, 0.4),
    (0.5, 0.5),
    (0.4, 0.6),
    (0.3, 0.7),
    (0.2, 0.8)
]

weight_results = []

for tfidf_weight, genre_weight in weights:
    scores = (
        tfidf_weight * df["tfidf_normalized"] +
        genre_weight * df["genre_normalized"]
    )

    ranked = scores.copy()
    ranked.loc[mixed_user_indices] = -1

    top_indices = ranked.nlargest(10).index

    overlaps = []

    for idx in top_indices:
        # Compare recommendation genres against the user's
        # aggregate preferred genres.
        item_genres = set(str(df.loc[idx, "genres"]).split("|"))
        matching_weight = sum(
            genre_preferences.get(g, 0)
            for g in item_genres
        )
        overlaps.append(matching_weight)

    weight_results.append({
        "tfidf_weight": tfidf_weight,
        "genre_weight": genre_weight,
        "avg_genre_score": round(sum(overlaps) / len(overlaps), 3)
    })

weight_results_df = pd.DataFrame(weight_results)

weight_results_df.sort_values(
    "avg_genre_score",
    ascending=False
)

,tfidf_weight,genre_weight,avg_genre_score
6,0.2,0.8,0.593
5,0.3,0.7,0.593
4,0.4,0.6,0.585
2,0.6,0.4,0.578
3,0.5,0.5,0.578
1,0.7,0.3,0.570
0,0.8,0.2,0.570


In [66]:
best_weights = weight_results_df.sort_values(
    "avg_genre_score",
    ascending=False
).iloc[0]

best_tfidf_weight = best_weights["tfidf_weight"]
best_genre_weight = best_weights["genre_weight"]

print("Best TF-IDF weight:", best_tfidf_weight)
print("Best genre weight:", best_genre_weight)

Best TF-IDF weight: 0.2
Best genre weight: 0.8


In [67]:
df["hybrid_score"] = (
    best_tfidf_weight * df["tfidf_normalized"] +
    best_genre_weight * df["genre_normalized"]
)

final_recommendations = (
    df[
        ~df.index.isin(mixed_user_indices)
    ][
        ["content_id", "title", "type", "genres", "hybrid_score"]
    ]
    .sort_values("hybrid_score", ascending=False)
    .head(10)
)

final_recommendations

,content_id,title,type,genres,hybrid_score
9,57911,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.859568
12,45952,Hunter x Hunter,anime,Sci-Fi & Fantasy|Action & Adventure|Animation|...,0.856073
17,65733,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.850990
5,57706,Ranma ½,anime,Comedy|Action & Adventure|Animation|Sci-Fi & F...,0.781546
27,31911,Fullmetal Alchemist: Brotherhood,anime,Animation|Action & Adventure|Sci-Fi & Fantasy|...,0.696731
15,209867,Frieren: Beyond Journey's End,anime,Animation|Action & Adventure|Drama|Sci-Fi & Fa...,0.695451
20,65942,Re:ZERO -Starting Life in Another World-,anime,Animation|Mystery|Action & Adventure|Sci-Fi & ...,0.686876
85,1315772,Minions & Monsters,movie,Adventure|Animation|Comedy|Family|Fantasy,0.673422
78,1084242,Zootopia 2,movie,Animation|Adventure|Comedy|Mystery|Family,0.663241
35,34860,Ninja Boy Rantaro,anime,Comedy|Action & Adventure|Kids|Animation|Family,0.655880


In [68]:
mixed_user_history = [
    "Supernatural",
    "Moana",
    "Fight Club",
    "Game of Thrones"
]

In [69]:
mixed_user_profile = np.asarray(
    weighted_tfidf_matrix[mixed_user_indices].mean(axis=0)
)

print("Mixed user profile shape:", mixed_user_profile.shape)

Mixed user profile shape: (1, 3618)


In [70]:
mixed_user_similarity = cosine_similarity(
    mixed_user_profile,
    weighted_tfidf_matrix
).flatten()

In [71]:
watched_indices = set(mixed_user_indices)

ranked_indices = np.argsort(mixed_user_similarity)[::-1]

mixed_recommendations = []

for i in ranked_indices:
    if i in watched_indices:
        continue

    mixed_recommendations.append({
        "content_id": df.loc[i, "content_id"],
        "title": df.loc[i, "title"],
        "type": df.loc[i, "type"],
        "score": round(mixed_user_similarity[i], 3)
    })

    if len(mixed_recommendations) == 10:
        break

pd.DataFrame(mixed_recommendations)

,content_id,title,type,score
0,31911,Fullmetal Alchemist: Brotherhood,anime,0.228
1,209867,Frieren: Beyond Journey's End,anime,0.225
2,45790,JoJo's Bizarre Adventure,anime,0.225
3,57706,Ranma ½,anime,0.217
4,220150,Pokémon Horizons,anime,0.206
5,65942,Re:ZERO -Starting Life in Another World-,anime,0.202
6,31910,Naruto Shippūden,anime,0.194
7,94664,Mushoku Tensei: Jobless Reincarnation,anime,0.193
8,1368337,The Odyssey,movie,0.191
9,57911,Doraemon,anime,0.188


In [72]:
mixed_user_indices = []

for title in mixed_user_history:
    matches = df.index[
        df["title"].str.lower() == title.lower()
    ]

    if len(matches) == 0:
        print(f"'{title}' not found")
    else:
        mixed_user_indices.append(matches[0])

print("Mixed user history:", mixed_user_history)
print("Mixed user indices:", mixed_user_indices)

print(
    df.loc[
        mixed_user_indices,
        ["title", "type", "genres"]
    ]
)

'Fight Club' not found
'Game of Thrones' not found
Mixed user history: ['Supernatural', 'Moana', 'Fight Club', 'Game of Thrones']
Mixed user indices: [56, 90]
           title   type                           genres
56  Supernatural     tv   Drama|Mystery|Sci-Fi & Fantasy
90         Moana  movie  Family|Fantasy|Comedy|Adventure


In [73]:
df[df["type"].isin(["movie", "tv"])][
    ["title", "type"]
].head(30)

,title,type
40,Running Man,tv
41,The Scandal,tv
42,Xplay,tv
43,The Mentalist,tv
44,Grey's Anatomy,tv
45,Law & Order,tv
46,The Office,tv
47,Late Night with Seth Meyers,tv
48,Come Home Love: Lo and Behold,tv
49,Watch What Happens Live with Andy Cohen,tv


In [74]:
movie_titles = df[df["type"] == "movie"]["title"].head(2).tolist()
tv_titles = df[df["type"] == "tv"]["title"].head(2).tolist()

mixed_user_history = movie_titles + tv_titles

print(mixed_user_history)

['Zootopia 2', 'The Novices', 'Running Man', 'The Scandal']


In [75]:
mixed_user_indices = []

for title in mixed_user_history:
    matches = df.index[
        df["title"].str.lower() == title.lower()
    ]

    if len(matches) > 0:
        mixed_user_indices.append(matches[0])

print(df.loc[
    mixed_user_indices,
    ["title", "type", "genres"]
])

          title   type                                     genres
78   Zootopia 2  movie  Animation|Adventure|Comedy|Mystery|Family
79  The Novices  movie                               Comedy|Drama
40  Running Man     tv                             Comedy|Reality
41  The Scandal     tv                                      Drama


In [76]:
mixed_user_history = [
    "Zootopia 2",
    "The Novices",
    "Running Man",
    "The Scandal"
]

mixed_user_indices = [
    df.index[df["title"].str.lower() == title.lower()][0]
    for title in mixed_user_history
]

print("Mixed user indices:", mixed_user_indices)

Mixed user indices: [78, 79, 40, 41]


In [77]:
mixed_user_profile = np.asarray(
    weighted_tfidf_matrix[mixed_user_indices].mean(axis=0)
)

mixed_user_similarity = cosine_similarity(
    mixed_user_profile,
    weighted_tfidf_matrix
).flatten()

In [78]:
watched_indices = set(mixed_user_indices)

ranked_indices = np.argsort(mixed_user_similarity)[::-1]

mixed_recommendations = []

for i in ranked_indices:
    if i in watched_indices:
        continue

    mixed_recommendations.append({
        "content_id": df.loc[i, "content_id"],
        "title": df.loc[i, "title"],
        "type": df.loc[i, "type"],
        "score": round(mixed_user_similarity[i], 3)
    })

    if len(mixed_recommendations) == 10:
        break

pd.DataFrame(mixed_recommendations)

,content_id,title,type,score
0,456,The Simpsons,tv,0.147
1,57775,Chibi Maruko-chan,anime,0.135
2,241002,Adam's Sweet Agony,anime,0.133
3,299536,Avengers: Infinity War,movie,0.126
4,1434,Family Guy,tv,0.122
5,113360,Jimihen!! Jimiko o Kae Chau Jun Isei Kouyuu,anime,0.122
6,1622,Supernatural,tv,0.121
7,91759,Come Home Love: Lo and Behold,tv,0.120
8,4304,Xplay,tv,0.118
9,154743,The Angel Next Door Spoils Me Rotten,anime,0.118


In [79]:
type_counts = df.loc[
    mixed_user_indices, "type"
].value_counts()

type_preferences = (
    type_counts / len(mixed_user_indices)
).to_dict()

print(type_preferences)

{'movie': 0.5, 'tv': 0.5}


In [80]:
def content_type_score(item_type, type_preferences):
    return type_preferences.get(item_type, 0)

df["type_score"] = df["type"].apply(
    lambda content_type: content_type_score(
        content_type,
        type_preferences
    )
)

print(
    df[
        ["title", "type", "type_score"]
    ].head(10)
)

                                               title   type  type_score
0                                     JUJUTSU KAISEN  anime         0.0
1  Secret Mission - Undercover Agents Never Back ...  anime         0.0
2                                      Dragon Ball Z  anime         0.0
3  Crazy Over His Fingers: Just the Two of Us in ...  anime         0.0
4                                           Ranma1/2  anime         0.0
5                                            Ranma ½  anime         0.0
6                                    Hunter x Hunter  anime         0.0
7                                        Atashin'chi  anime         0.0
8               The Angel Next Door Spoils Me Rotten  anime         0.0
9                                           Doraemon  anime         0.0


In [82]:
df["type_score"].value_counts()

type_score
0.5    78
0.0    40
Name: count, dtype: int64

In [83]:
df["final_score"] = (
    0.20 * df["tfidf_normalized"] +
    0.40 * df["genre_normalized"] +
    0.40 * df["type_score"]
)

final_recommendations = (
    df[
        ~df.index.isin(mixed_user_indices)
    ][
        [
            "content_id",
            "title",
            "type",
            "genres",
            "tfidf_normalized",
            "genre_normalized",
            "type_score",
            "final_score"
        ]
    ]
    .sort_values("final_score", ascending=False)
    .head(15)
)

final_recommendations

,content_id,title,type,genres,tfidf_normalized,genre_normalized,type_score,final_score
90,1108427,Moana,movie,Family|Fantasy|Comedy|Adventure,0.950441,0.555556,0.5,0.612311
56,1622,Supernatural,tv,Drama|Mystery|Sci-Fi & Fantasy,1.000000,0.444444,0.5,0.577778
85,1315772,Minions & Monsters,movie,Adventure|Animation|Comedy|Family|Fantasy,0.255999,0.777778,0.5,0.562311
109,1084244,Toy Story 5,movie,Animation|Family|Comedy|Adventure,0.216310,0.666667,0.5,0.509929
10,37854,One Piece,anime,Action & Adventure|Comedy|Animation,0.981810,0.666667,0.0,0.463029
9,57911,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.297841,1.000000,0.0,0.459568
12,45952,Hunter x Hunter,anime,Sci-Fi & Fantasy|Action & Adventure|Animation|...,0.280366,1.000000,0.0,0.456073
2,12971,Dragon Ball Z,anime,Animation|Sci-Fi & Fantasy|Action & Adventure,0.929375,0.666667,0.0,0.452542
17,65733,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.254949,1.000000,0.0,0.450990
60,1434,Family Guy,tv,Animation|Comedy,0.247286,0.444444,0.5,0.427235


In [84]:
final_recommendations = (
    df[
        ~df.index.isin(mixed_user_indices)
    ][
        [
            "content_id",
            "title",
            "type",
            "genres",
            "tfidf_normalized",
            "genre_normalized",
            "type_score",
            "final_score"
        ]
    ]
    .sort_values("final_score", ascending=False)
    .head(15)
)

final_recommendations

,content_id,title,type,genres,tfidf_normalized,genre_normalized,type_score,final_score
90,1108427,Moana,movie,Family|Fantasy|Comedy|Adventure,0.950441,0.555556,0.5,0.612311
56,1622,Supernatural,tv,Drama|Mystery|Sci-Fi & Fantasy,1.000000,0.444444,0.5,0.577778
85,1315772,Minions & Monsters,movie,Adventure|Animation|Comedy|Family|Fantasy,0.255999,0.777778,0.5,0.562311
109,1084244,Toy Story 5,movie,Animation|Family|Comedy|Adventure,0.216310,0.666667,0.5,0.509929
10,37854,One Piece,anime,Action & Adventure|Comedy|Animation,0.981810,0.666667,0.0,0.463029
9,57911,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.297841,1.000000,0.0,0.459568
12,45952,Hunter x Hunter,anime,Sci-Fi & Fantasy|Action & Adventure|Animation|...,0.280366,1.000000,0.0,0.456073
2,12971,Dragon Ball Z,anime,Animation|Sci-Fi & Fantasy|Action & Adventure,0.929375,0.666667,0.0,0.452542
17,65733,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.254949,1.000000,0.0,0.450990
60,1434,Family Guy,tv,Animation|Comedy,0.247286,0.444444,0.5,0.427235


In [85]:
print("Watched indices:", mixed_user_indices)

print(
    df.loc[
        mixed_user_indices,
        ["title", "type"]
    ]
)

Watched indices: [78, 79, 40, 41]
          title   type
78   Zootopia 2  movie
79  The Novices  movie
40  Running Man     tv
41  The Scandal     tv


In [86]:
remaining_df = df[
    ~df.index.isin(mixed_user_indices)
]

print(
    remaining_df.loc[
        remaining_df["title"].isin(mixed_user_history),
        ["title", "type"]
    ]
)

Empty DataFrame
Columns: [title, type]
Index: []


In [87]:
final_recommendations = (
    remaining_df[
        [
            "content_id",
            "title",
            "type",
            "genres",
            "tfidf_normalized",
            "genre_normalized",
            "type_score",
            "final_score"
        ]
    ]
    .sort_values("final_score", ascending=False)
    .head(15)
)

final_recommendations

,content_id,title,type,genres,tfidf_normalized,genre_normalized,type_score,final_score
90,1108427,Moana,movie,Family|Fantasy|Comedy|Adventure,0.950441,0.555556,0.5,0.612311
56,1622,Supernatural,tv,Drama|Mystery|Sci-Fi & Fantasy,1.000000,0.444444,0.5,0.577778
85,1315772,Minions & Monsters,movie,Adventure|Animation|Comedy|Family|Fantasy,0.255999,0.777778,0.5,0.562311
109,1084244,Toy Story 5,movie,Animation|Family|Comedy|Adventure,0.216310,0.666667,0.5,0.509929
10,37854,One Piece,anime,Action & Adventure|Comedy|Animation,0.981810,0.666667,0.0,0.463029
9,57911,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.297841,1.000000,0.0,0.459568
12,45952,Hunter x Hunter,anime,Sci-Fi & Fantasy|Action & Adventure|Animation|...,0.280366,1.000000,0.0,0.456073
2,12971,Dragon Ball Z,anime,Animation|Sci-Fi & Fantasy|Action & Adventure,0.929375,0.666667,0.0,0.452542
17,65733,Doraemon,anime,Action & Adventure|Animation|Comedy|Sci-Fi & F...,0.254949,1.000000,0.0,0.450990
60,1434,Family Guy,tv,Animation|Comedy,0.247286,0.444444,0.5,0.427235


In [88]:
# Current user history
mixed_user_history = [
    "Zootopia 2",
    "The Novices",
    "Running Man",
    "The Scandal"
]

# Find their indices
mixed_user_indices = []

for title in mixed_user_history:
    matches = df.index[
        df["title"].str.lower() == title.lower()
    ]

    if len(matches) > 0:
        mixed_user_indices.append(matches[0])

# Verify history
print("USER HISTORY:")
print(
    df.loc[
        mixed_user_indices,
        ["title", "type"]
    ]
)

# Build user profile
mixed_user_profile = np.asarray(
    weighted_tfidf_matrix[mixed_user_indices].mean(axis=0)
)

# Calculate TF-IDF similarity
mixed_user_similarity = cosine_similarity(
    mixed_user_profile,
    weighted_tfidf_matrix
).flatten()

# Update TF-IDF normalized score
df["tfidf_normalized"] = MinMaxScaler().fit_transform(
    mixed_user_similarity.reshape(-1, 1)
).flatten()

# Build genre preferences
user_genres = []

for i in mixed_user_indices:
    user_genres.extend(
        str(df.loc[i, "genres"]).split("|")
    )

genre_counts = Counter(user_genres)

total_genres = sum(genre_counts.values())

genre_preferences = {
    genre: count / total_genres
    for genre, count in genre_counts.items()
}

# Genre score
df["genre_score"] = df["genres"].apply(
    lambda genres: genre_preference_score(
        genres,
        genre_preferences
    )
)

df["genre_normalized"] = MinMaxScaler().fit_transform(
    df[["genre_score"]]
).flatten()

# Content-type preferences
type_counts = df.loc[
    mixed_user_indices, "type"
].value_counts()

type_preferences = (
    type_counts / len(mixed_user_indices)
).to_dict()

df["type_score"] = df["type"].apply(
    lambda content_type: type_preferences.get(
        content_type, 0
    )
)

# Final score
df["final_score"] = (
    0.20 * df["tfidf_normalized"] +
    0.40 * df["genre_normalized"] +
    0.40 * df["type_score"]
)

# IMPORTANT: exclude current history
recommendation_df = df[
    ~df.index.isin(mixed_user_indices)
].copy()

# Final recommendations
final_recommendations = (
    recommendation_df[
        [
            "content_id",
            "title",
            "type",
            "genres",
            "tfidf_normalized",
            "genre_normalized",
            "type_score",
            "final_score"
        ]
    ]
    .sort_values(
        "final_score",
        ascending=False
    )
    .head(15)
)

final_recommendations

USER HISTORY:
          title   type
78   Zootopia 2  movie
79  The Novices  movie
40  Running Man     tv
41  The Scandal     tv


,content_id,title,type,genres,tfidf_normalized,genre_normalized,type_score,final_score
48,91759,Come Home Love: Lo and Behold,tv,Family|Comedy|Drama,0.186557,0.857143,0.5,0.580169
109,1084244,Toy Story 5,movie,Animation|Family|Comedy|Adventure,0.141121,0.857143,0.5,0.571081
85,1315772,Minions & Monsters,movie,Adventure|Animation|Comedy|Family|Fantasy,0.141083,0.857143,0.5,0.571074
66,14424,Young Hearts,tv,Soap|Comedy|Drama,0.167993,0.714286,0.5,0.519313
104,1248832,Digger,movie,Comedy|Drama,0.151430,0.714286,0.5,0.516000
90,1108427,Moana,movie,Family|Fantasy|Comedy|Adventure,0.134428,0.714286,0.5,0.512600
76,79744,The Rookie,tv,Crime|Drama|Comedy,0.124182,0.714286,0.5,0.510551
115,1538457,Meesaya Murukku 2,movie,Comedy|Romance|Drama,0.113035,0.714286,0.5,0.508321
97,1204680,Coyote vs. Acme,movie,Comedy|Adventure|Family,0.093192,0.714286,0.5,0.504353
74,97546,Ted Lasso,tv,Drama|Comedy,0.087293,0.714286,0.5,0.503173


In [89]:
diverse_recommendations = []

type_counts = {
    "movie": 0,
    "tv": 0,
    "anime": 0
}

max_per_type = 5

for idx, row in recommendation_df.sort_values(
    "final_score",
    ascending=False
).iterrows():

    content_type = row["type"]

    if type_counts[content_type] >= max_per_type:
        continue

    diverse_recommendations.append({
        "content_id": row["content_id"],
        "title": row["title"],
        "type": content_type,
        "score": round(row["final_score"], 3)
    })

    type_counts[content_type] += 1

    if len(diverse_recommendations) == 10:
        break

diverse_recommendations = pd.DataFrame(
    diverse_recommendations
)

diverse_recommendations

,content_id,title,type,score
0,91759,Come Home Love: Lo and Behold,tv,0.580
1,1084244,Toy Story 5,movie,0.571
2,1315772,Minions & Monsters,movie,0.571
3,14424,Young Hearts,tv,0.519
4,1248832,Digger,movie,0.516
5,1108427,Moana,movie,0.513
6,79744,The Rookie,tv,0.511
7,1538457,Meesaya Murukku 2,movie,0.508
8,97546,Ted Lasso,tv,0.503
9,456,The Simpsons,tv,0.476


In [90]:
import os

os.getcwd()

'E:\\Engineering\\CinePlan\\ml\\notebooks'

In [91]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

joblib.dump(
    weighted_vectorizer,
    "../models/tfidf_vectorizer.pkl"
)

joblib.dump(
    weighted_tfidf_matrix,
    "../models/tfidf_matrix.pkl"
)

df.to_pickle(
    "../models/content_data.pkl"
)

print("Model artifacts saved successfully.")

Model artifacts saved successfully.


In [92]:
os.listdir("../models")

['content_data.pkl', 'tfidf_matrix.pkl', 'tfidf_vectorizer.pkl']

In [93]:
import pandas as pd

df = pd.read_csv("../data/raw/content.csv")

print("Dataset shape:", df.shape)
print(df["type"].value_counts())

Dataset shape: (813, 10)
type
movie    296
anime    267
tv       250
Name: count, dtype: int64


In [94]:
text_columns = ["title", "overview", "genres", "cast"]

df[text_columns] = df[text_columns].fillna("")

df["content_text"] = df[text_columns].agg(" ".join, axis=1)

In [95]:
df["content_text_weighted"] = (
    df["title"] + " " +
    df["overview"] + " " +
    df["overview"] + " " +
    df["genres"] + " " +
    df["genres"] + " " +
    df["genres"] + " " +
    df["cast"]
)

In [96]:
from sklearn.feature_extraction.text import TfidfVectorizer

weighted_vectorizer = TfidfVectorizer()

weighted_tfidf_matrix = weighted_vectorizer.fit_transform(
    df["content_text_weighted"]
)

print(weighted_tfidf_matrix.shape)

(813, 13390)


In [97]:
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(weighted_tfidf_matrix)

print(similarity_matrix.shape)

(813, 813)


In [98]:
mixed_user_history = [
    "Zootopia 2",
    "The Novices",
    "Running Man",
    "The Scandal"
]

mixed_user_indices = []

for title in mixed_user_history:
    matches = df.index[df["title"] == title].tolist()

    if matches:
        mixed_user_indices.append(matches[0])
    else:
        print(f"Not found: {title}")

print("Indices:", mixed_user_indices)

df.loc[mixed_user_indices, ["title", "type", "genres"]]

Indices: [532, 531, 284, 290]


,title,type,genres
532,Zootopia 2,movie,Animation|Adventure|Comedy|Mystery|Family
531,The Novices,movie,Comedy|Drama
284,Running Man,tv,Comedy|Reality
290,The Scandal,tv,Drama


In [99]:
import numpy as np

user_profile = np.asarray(
    weighted_tfidf_matrix[mixed_user_indices].mean(axis=0)
)

print("User profile shape:", user_profile.shape)

User profile shape: (1, 13390)


In [100]:
from sklearn.metrics.pairwise import cosine_similarity

user_scores = cosine_similarity(
    user_profile,
    weighted_tfidf_matrix
).flatten()

print("Scores shape:", user_scores.shape)

Scores shape: (813,)


In [101]:
df["tfidf_score"] = user_scores

print(df["tfidf_score"].describe())

count    813.000000
mean       0.062891
std        0.039116
min        0.000000
25%        0.046934
50%        0.059798
75%        0.073523
max        0.537799
Name: tfidf_score, dtype: float64


In [102]:
from sklearn.preprocessing import MinMaxScaler

df["tfidf_normalized"] = MinMaxScaler().fit_transform(
    df[["tfidf_score"]]
).flatten()
count    813.000000
mean       0.116941
std        0.072734
min        0.000000
25%        0.087271
50%        0.111189
75%        0.136712
max        1.000000
Name: tfidf_normalized, dtype: float64
print(df["tfidf_normalized"].describe())

count    813.000000
mean       0.116941
std        0.072734
min        0.000000
25%        0.087271
50%        0.111189
75%        0.136712
max        1.000000
Name: tfidf_normalized, dtype: float64


In [103]:
from sklearn.preprocessing import MinMaxScaler

df["tfidf_normalized"] = MinMaxScaler().fit_transform(
    df[["tfidf_score"]]
).flatten()

print(df["tfidf_normalized"].describe())

count    813.000000
mean       0.116941
std        0.072734
min        0.000000
25%        0.087271
50%        0.111189
75%        0.136712
max        1.000000
Name: tfidf_normalized, dtype: float64


In [104]:
from collections import Counter

user_genres = []

for i in mixed_user_indices:
    user_genres.extend(
        str(df.loc[i, "genres"]).split("|")
    )

genre_counts = Counter(user_genres)

total_genres = sum(genre_counts.values())

genre_preferences = {
    genre: count / total_genres
    for genre, count in genre_counts.items()
}

print("Genre counts:")
print(genre_counts)

print("\nGenre preferences:")
print(genre_preferences)

Genre counts:
Counter({'Comedy': 3, 'Drama': 2, 'Animation': 1, 'Adventure': 1, 'Mystery': 1, 'Family': 1, 'Reality': 1})

Genre preferences:
{'Animation': 0.1, 'Adventure': 0.1, 'Comedy': 0.3, 'Mystery': 0.1, 'Family': 0.1, 'Drama': 0.2, 'Reality': 0.1}


In [105]:
def genre_preference_score(item_genres, genre_preferences):
    genres = set(str(item_genres).split("|"))

    return sum(
        genre_preferences.get(genre, 0)
        for genre in genres
    )


df["genre_score"] = df["genres"].apply(
    lambda genres: genre_preference_score(
        genres,
        genre_preferences
    )
)

print(df["genre_score"].describe())

count    813.000000
mean       0.244280
std        0.172013
min        0.000000
25%        0.100000
50%        0.200000
75%        0.400000
max        0.800000
Name: genre_score, dtype: float64


In [106]:
df["genre_normalized"] = MinMaxScaler().fit_transform(
    df[["genre_score"]]
).flatten()

print(df["genre_normalized"].describe())

count    813.000000
mean       0.305351
std        0.215017
min        0.000000
25%        0.125000
50%        0.250000
75%        0.500000
max        1.000000
Name: genre_normalized, dtype: float64


In [107]:
type_counts = df.loc[
    mixed_user_indices, "type"
].value_counts()

type_preferences = (
    type_counts / len(mixed_user_indices)
).to_dict()

print("Type counts:")
print(type_counts)

print("\nType preferences:")
print(type_preferences)


Type counts:
type
movie    2
tv       2
Name: count, dtype: int64

Type preferences:
{'movie': 0.5, 'tv': 0.5}


In [108]:
def content_type_score(item_type, type_preferences):
    return type_preferences.get(item_type, 0)


df["type_score"] = df["type"].apply(
    lambda content_type: content_type_score(
        content_type,
        type_preferences
    )
)

print(df["type_score"].value_counts())

type_score
0.5    546
0.0    267
Name: count, dtype: int64


In [109]:
df["final_score"] = (
    0.20 * df["tfidf_normalized"] +
    0.40 * df["genre_normalized"] +
    0.40 * df["type_score"]
)

print(df["final_score"].describe())

count    813.000000
mean       0.279846
std        0.117490
min        0.052195
25%        0.215674
50%        0.274748
75%        0.365621
max        0.739750
Name: final_score, dtype: float64


In [110]:
recommendation_df = df.drop(
    index=mixed_user_indices
).copy()

print("Candidates after history exclusion:", len(recommendation_df))

Candidates after history exclusion: 809


In [111]:
recommendation_df = recommendation_df.sort_values(
    "final_score",
    ascending=False
)

recommendation_df[
    ["title", "type", "genres", "final_score"]
].head(10)

,title,type,genres,final_score
753,Inside Out,movie,Animation|Family|Adventure|Drama|Comedy,0.627013
337,Desperate Housewives,tv,Mystery|Drama|Comedy,0.541234
273,Uusi Päivä,tv,Comedy|Drama|Family,0.539319
492,Rizzoli & Isles,tv,Crime|Drama|Mystery|Comedy,0.531807
555,Shrek the Third,movie,Fantasy|Adventure|Animation|Comedy|Family,0.531696
515,Come Home Love: Lo and Behold,tv,Family|Comedy|Drama,0.531067
417,Young Sheldon,tv,Comedy|Family|Drama,0.529782
382,Charmed,tv,Comedy|Drama|Mystery|Sci-Fi & Fantasy,0.528196
562,Shrek Forever After,movie,Comedy|Adventure|Fantasy|Animation|Family,0.528080
681,Moana 2,movie,Adventure|Animation|Comedy|Family|Fantasy,0.527898


In [112]:
diverse_recommendations = []

type_counts = {
    "movie": 0,
    "tv": 0,
    "anime": 0
}

max_per_type = 5

for idx, row in recommendation_df.iterrows():
    content_type = row["type"]

    if type_counts[content_type] >= max_per_type:
        continue

    diverse_recommendations.append({
        "content_id": row["content_id"],
        "title": row["title"],
        "type": row["type"],
        "score": round(row["final_score"], 3)
    })

    type_counts[content_type] += 1

    if len(diverse_recommendations) == 10:
        break

diverse_recommendations = pd.DataFrame(
    diverse_recommendations
)

diverse_recommendations

,content_id,title,type,score
0,150540,Inside Out,movie,0.627
1,693,Desperate Housewives,tv,0.541
2,54999,Uusi Päivä,tv,0.539
3,32895,Rizzoli & Isles,tv,0.532
4,810,Shrek the Third,movie,0.532
5,91759,Come Home Love: Lo and Behold,tv,0.531
6,71728,Young Sheldon,tv,0.530
7,10192,Shrek Forever After,movie,0.528
8,1241982,Moana 2,movie,0.528
9,1022789,Inside Out 2,movie,0.527


In [113]:
import os
import pickle

os.makedirs("../models", exist_ok=True)

In [114]:
with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(weighted_vectorizer, f)

with open("../models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(weighted_tfidf_matrix, f)

with open("../models/content_data.pkl", "wb") as f:
    pickle.dump(df, f)

print("Model artifacts saved successfully.")

Model artifacts saved successfully.


In [115]:
import os

print(os.getcwd())

E:\Engineering\CinePlan\ml\notebooks


In [116]:
import os
import pickle

os.makedirs("../models", exist_ok=True)

with open("../models/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(weighted_vectorizer, f)

with open("../models/tfidf_matrix.pkl", "wb") as f:
    pickle.dump(weighted_tfidf_matrix, f)

with open("../models/content_data.pkl", "wb") as f:
    pickle.dump(df, f)

print("Model artifacts saved successfully.")

Model artifacts saved successfully.


In [117]:
for filename in [
    "../models/tfidf_vectorizer.pkl",
    "../models/tfidf_matrix.pkl",
    "../models/content_data.pkl"
]:
    print(filename, "->", os.path.exists(filename))

../models/tfidf_vectorizer.pkl -> True
../models/tfidf_matrix.pkl -> True
../models/content_data.pkl -> True


In [118]:
import pickle

with open("../models/tfidf_vectorizer.pkl", "rb") as f:
    loaded_vectorizer = pickle.load(f)

with open("../models/tfidf_matrix.pkl", "rb") as f:
    loaded_matrix = pickle.load(f)

with open("../models/content_data.pkl", "rb") as f:
    loaded_df = pickle.load(f)

print("Matrix shape:", loaded_matrix.shape)
print("Dataset shape:", loaded_df.shape)

print("\nContent distribution:")
print(loaded_df["type"].value_counts())

Matrix shape: (813, 13390)
Dataset shape: (813, 18)

Content distribution:
type
movie    296
anime    267
tv       250
Name: count, dtype: int64


In [119]:
assert loaded_matrix.shape[0] == len(loaded_df)
assert loaded_matrix.shape[1] == len(loaded_vectorizer.vocabulary_)

print("Artifact verification successful ✅")

Artifact verification successful ✅
